# Notebook 4 — CIFAR Reviewer-Killer Robustness

This notebook closes the remaining **CIFAR robustness** reviewer gaps for the RAC paper.

It does **not** regenerate diffusion samples. It reuses the already-saved CIFAR tensors and runs only small correction networks.

It tests three questions:

1. **Data-size robustness**  
   Does the C=500 vs C=50 residual-difficulty gap persist as the training set grows?

2. **Architecture robustness**  
   Does the gap persist across local, dilated, and residual CNN correctors?

3. **Capacity robustness**  
   Does the gap persist when correction capacity increases by ~two orders of magnitude?

Finally, it derives **RAC threshold sensitivity** from the capacity experiment.

**This no-matplotlib version intentionally creates numerical CSV outputs only. Figures can be generated later from the saved CSVs in a clean lightweight session.**

## Important
Every numerical result is computed from the actual saved tensors.  
Every training row is checkpointed to CSV immediately, so if the kernel dies, rerunning the notebook skips finished jobs.

In [1]:
# ============================================================
# GLOBAL IMPORTS + CONFIG
# ============================================================

from pathlib import Path
import random
import copy
import json
import math
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader



SEED_BASE = 20260928

random.seed(SEED_BASE)
np.random.seed(SEED_BASE)
torch.manual_seed(SEED_BASE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_BASE)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Absolute path avoids Windows/Jupyter working-directory issues.
PROJECT_DIR = Path(
    r"D:\PhD EMP\Paper Writting\STEP_COMPLEXITY"
)

CIFAR_DIR = PROJECT_DIR / "ddim_residual_data_512"

# Reuse legacy Step-5 cache if it already exists.
LEGACY_DIR = PROJECT_DIR / "step5_robustness"
OUT_DIR = PROJECT_DIR / "step5_robustness"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset split
N_TOTAL = 512
N_TRAIN = 384
N_VAL = 64
N_TEST = 64

TRAIN_END = 384
VAL_END = 448

# Training defaults
LR = 3e-4
BATCH_SIZE = 32
MAX_EPOCHS = 1000
PATIENCE = 60

print("device:", device)
print("PROJECT_DIR exists:", PROJECT_DIR.exists())
print("CIFAR_DIR:", CIFAR_DIR)
print("OUT_DIR:", OUT_DIR)

device: cpu
PROJECT_DIR exists: True
CIFAR_DIR: D:\PhD EMP\Paper Writting\STEP_COMPLEXITY\ddim_residual_data_512
OUT_DIR: D:\PhD EMP\Paper Writting\STEP_COMPLEXITY\step5_robustness


## 1. Load the fixed CIFAR tensors

In [2]:
required = [
    CIFAR_DIR / "z.pt",
    CIFAR_DIR / "teacher_1000.pt",
    CIFAR_DIR / "baseline_50.pt",
    CIFAR_DIR / "baseline_500.pt",
]

for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

z = torch.load(CIFAR_DIR / "z.pt", map_location="cpu").float()
teacher = torch.load(CIFAR_DIR / "teacher_1000.pt", map_location="cpu").float()

baselines = {
    50: torch.load(CIFAR_DIR / "baseline_50.pt", map_location="cpu").float(),
    500: torch.load(CIFAR_DIR / "baseline_500.pt", map_location="cpu").float(),
}

assert len(z) == 512
assert len(teacher) == 512

z_val = z[384:448]
z_test = z[448:512]

residuals = {
    C: teacher - B
    for C, B in baselines.items()
}

print("Loaded CIFAR tensors.")
print("z:", z.shape)
print("teacher:", teacher.shape)

Loaded CIFAR tensors.
z: torch.Size([512, 3, 32, 32])
teacher: torch.Size([512, 3, 32, 32])


## 2. Correction architectures

In [3]:
class LocalCorrectionCNN(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, 3, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


class DilatedCorrectionCNN(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1, dilation=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=2, dilation=2),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=4, dilation=4),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1, dilation=1),
            nn.GELU(),
            nn.Conv2d(width, 3, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


class ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.block(x))


class ResCorrectionCNN(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.in_proj = nn.Conv2d(3, width, 3, padding=1)
        self.body = nn.Sequential(
            ResidualBlock(width),
            ResidualBlock(width),
            ResidualBlock(width),
        )
        self.out_proj = nn.Conv2d(width, 3, 3, padding=1)

    def forward(self, x):
        x = self.in_proj(x)
        x = self.body(x)
        return self.out_proj(x)


ARCH_FACTORIES = {
    "local": LocalCorrectionCNN,
    "dilated": DilatedCorrectionCNN,
    "resnet": ResCorrectionCNN,
}


def count_params(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


print("Architecture parameter counts:")
for arch, factory in ARCH_FACTORIES.items():
    m = factory(16)
    print(arch, count_params(m))
    del m

Architecture parameter counts:
local 5523
dilated 7843
resnet 14803


## 3. Shared robust trainer

In [4]:
def _metrics(pred, target):
    mse = ((pred - target) ** 2).mean().item()
    energy = (target ** 2).mean().item()
    nmse = mse / energy

    mae = (pred - target).abs().mean().item()
    target_abs = target.abs().mean().item()
    nmae = mae / target_abs

    p = pred.flatten(1)
    t = target.flatten(1)

    cosine = F.cosine_similarity(
        p,
        t,
        dim=1,
        eps=1e-8,
    ).mean().item()

    return {
        "nmse": nmse,
        "nmae": nmae,
        "cosine": cosine,
    }


def train_corrector(
    z_train,
    r_train,
    z_val,
    r_val,
    z_test,
    r_test,
    width,
    architecture="local",
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    patience=PATIENCE,
    seed=0,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = ARCH_FACTORIES[architecture](width).to(device)

    train_energy = (r_train ** 2).mean().item()
    if train_energy <= 1e-14:
        raise ValueError("Residual energy is effectively zero.")

    train_rms = train_energy ** 0.5
    y_train = r_train / train_rms

    g = torch.Generator().manual_seed(seed)

    loader = DataLoader(
        TensorDataset(z_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        generator=g,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
    )

    loss_fn = nn.MSELoss()

    val_energy = (r_val ** 2).mean().item()

    best_val_nmse = float("inf")
    best_state = None
    best_epoch = 0
    bad_epochs = 0

    for epoch in range(max_epochs):
        model.train()

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)

            if not torch.isfinite(loss):
                raise ValueError("Non-finite training loss.")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )
            optimizer.step()

        model.eval()

        with torch.no_grad():
            pred_val = (
                model(z_val.to(device)).cpu()
                * train_rms
            )

        val_nmse = (
            ((pred_val - r_val) ** 2).mean().item()
            / val_energy
        )

        if val_nmse < best_val_nmse - 1e-5:
            best_val_nmse = val_nmse
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_train = (
            model(z_train.to(device)).cpu()
            * train_rms
        )
        pred_test = (
            model(z_test.to(device)).cpu()
            * train_rms
        )

    train_metrics = _metrics(pred_train, r_train)
    test_metrics = _metrics(pred_test, r_test)

    result = {
        "architecture": architecture,
        "width": width,
        "params": count_params(model),
        "seed": seed,
        "best_epoch": best_epoch,
        "train_rms": train_rms,
        "train_nmse": train_metrics["nmse"],
        "test_nmse": test_metrics["nmse"],
        "test_nmae": test_metrics["nmae"],
        "test_cosine": test_metrics["cosine"],
        "val_nmse": best_val_nmse,
    }

    del model
    gc.collect()

    if device == "cuda":
        torch.cuda.empty_cache()

    return result

# 4A. Training-data robustness

18 jobs total if no cache exists:

\[
2\ \text{baselines}
\times 3\ \text{train sizes}
\times 3\ \text{seeds}
= 18.
\]

Results are saved after every completed training.

In [5]:
TRAIN_SIZES = [96, 192, 384]
DATA_SEEDS = [0, 1, 2]
DATA_WIDTH = 8

data_path = OUT_DIR / "cifar_data_size_robustness.csv"

if data_path.exists():
    data_results = pd.read_csv(data_path).to_dict("records")
    print(f"Loaded {len(data_results)} cached data-size rows.")
else:
    data_results = []

def data_done(C, n_train, seed):
    return any(
        int(r["C"]) == C
        and int(r["n_train"]) == n_train
        and int(r["seed"]) == seed
        for r in data_results
    )

for C in [50, 500]:
    R = residuals[C]
    r_val = R[384:448]
    r_test = R[448:512]

    for n_train in TRAIN_SIZES:
        z_train = z[:n_train]
        r_train = R[:n_train]

        for seed in DATA_SEEDS:
            if data_done(C, n_train, seed):
                print("SKIP DATA", C, n_train, seed)
                continue

            print("RUN DATA", C, n_train, seed)

            out = train_corrector(
                z_train,
                r_train,
                z_val,
                r_val,
                z_test,
                r_test,
                width=DATA_WIDTH,
                architecture="local",
                seed=seed,
            )

            out["C"] = C
            out["n_train"] = n_train

            data_results.append(out)

            pd.DataFrame(data_results).to_csv(
                data_path,
                index=False,
            )

data_df = pd.DataFrame(data_results)

data_summary = (
    data_df
    .groupby(["C", "n_train"], as_index=False)
    .agg(
        nmse_mean=("test_nmse", "mean"),
        nmse_std=("test_nmse", "std"),
    )
)

data_pivot = data_summary.pivot(
    index="n_train",
    columns="C",
    values="nmse_mean",
)

data_pivot["delta_500_minus_50"] = (
    data_pivot[500] - data_pivot[50]
)

print(data_pivot)

data_pivot.to_csv(
    OUT_DIR / "cifar_data_size_gap.csv"
)

Loaded 18 cached data-size rows.
SKIP DATA 50 96 0
SKIP DATA 50 96 1
SKIP DATA 50 96 2
SKIP DATA 50 192 0
SKIP DATA 50 192 1
SKIP DATA 50 192 2
SKIP DATA 50 384 0
SKIP DATA 50 384 1
SKIP DATA 50 384 2
SKIP DATA 500 96 0
SKIP DATA 500 96 1
SKIP DATA 500 96 2
SKIP DATA 500 192 0
SKIP DATA 500 192 1
SKIP DATA 500 192 2
SKIP DATA 500 384 0
SKIP DATA 500 384 1
SKIP DATA 500 384 2
C              50       500  delta_500_minus_50
n_train                                        
96       0.329442  0.559254            0.229812
192      0.328217  0.556512            0.228296
384      0.321530  0.551832            0.230302


# 4B. Architecture robustness

18 jobs total if no cache exists:

\[
2\ \text{baselines}
\times 3\ \text{architectures}
\times 3\ \text{seeds}
= 18.
\]

In [6]:
ARCHITECTURES = ["local", "dilated", "resnet"]
ARCH_SEEDS = [0, 1, 2]
ARCH_WIDTH = 16

arch_path = OUT_DIR / "cifar_architecture_robustness.csv"

if arch_path.exists():
    arch_results = pd.read_csv(arch_path).to_dict("records")
    print(f"Loaded {len(arch_results)} cached architecture rows.")
else:
    arch_results = []

def arch_done(C, architecture, seed):
    return any(
        int(r["C"]) == C
        and r["architecture"] == architecture
        and int(r["seed"]) == seed
        for r in arch_results
    )

z_train = z[:384]

for C in [50, 500]:
    R = residuals[C]

    r_train = R[:384]
    r_val = R[384:448]
    r_test = R[448:512]

    for architecture in ARCHITECTURES:
        for seed in ARCH_SEEDS:
            if arch_done(C, architecture, seed):
                print("SKIP ARCH", C, architecture, seed)
                continue

            print("RUN ARCH", C, architecture, seed)

            out = train_corrector(
                z_train,
                r_train,
                z_val,
                r_val,
                z_test,
                r_test,
                width=ARCH_WIDTH,
                architecture=architecture,
                seed=seed,
            )

            out["C"] = C
            arch_results.append(out)

            pd.DataFrame(arch_results).to_csv(
                arch_path,
                index=False,
            )

arch_df = pd.DataFrame(arch_results)

arch_summary = (
    arch_df
    .groupby(
        ["architecture", "C", "params"],
        as_index=False,
    )
    .agg(
        nmse_mean=("test_nmse", "mean"),
        nmse_std=("test_nmse", "std"),
    )
)

arch_gap = arch_summary.pivot_table(
    index="architecture",
    columns="C",
    values="nmse_mean",
)

arch_gap["delta_500_minus_50"] = (
    arch_gap[500] - arch_gap[50]
)

print(arch_gap)

arch_gap.to_csv(
    OUT_DIR / "cifar_architecture_gap.csv"
)

Loaded 18 cached architecture rows.
SKIP ARCH 50 local 0
SKIP ARCH 50 local 1
SKIP ARCH 50 local 2
SKIP ARCH 50 dilated 0
SKIP ARCH 50 dilated 1
SKIP ARCH 50 dilated 2
SKIP ARCH 50 resnet 0
SKIP ARCH 50 resnet 1
SKIP ARCH 50 resnet 2
SKIP ARCH 500 local 0
SKIP ARCH 500 local 1
SKIP ARCH 500 local 2
SKIP ARCH 500 dilated 0
SKIP ARCH 500 dilated 1
SKIP ARCH 500 dilated 2
SKIP ARCH 500 resnet 0
SKIP ARCH 500 resnet 1
SKIP ARCH 500 resnet 2
C                   50       500  delta_500_minus_50
architecture                                        
dilated       0.298916  0.533599            0.234683
local         0.320659  0.552337            0.231678
resnet        0.309351  0.544975            0.235625


# 4C. Capacity robustness

18 jobs total if no cache exists:

\[
2\ \text{baselines}
\times 3\ \text{widths}
\times 3\ \text{seeds}
= 18.
\]

In [7]:
CAP_WIDTHS = [8, 32, 128]
CAP_SEEDS = [0, 1, 2]

cap_path = OUT_DIR / "cifar_capacity_robustness.csv"

if cap_path.exists():
    cap_results = pd.read_csv(cap_path).to_dict("records")
    print(f"Loaded {len(cap_results)} cached capacity rows.")
else:
    cap_results = []

def cap_done(C, width, seed):
    return any(
        int(r["C"]) == C
        and int(r["width"]) == width
        and int(r["seed"]) == seed
        for r in cap_results
    )

for C in [50, 500]:
    R = residuals[C]

    r_train = R[:384]
    r_val = R[384:448]
    r_test = R[448:512]

    for width in CAP_WIDTHS:
        for seed in CAP_SEEDS:
            if cap_done(C, width, seed):
                print("SKIP CAP", C, width, seed)
                continue

            print("RUN CAP", C, width, seed)

            out = train_corrector(
                z[:384],
                r_train,
                z_val,
                r_val,
                z_test,
                r_test,
                width=width,
                architecture="local",
                seed=seed,
            )

            out["C"] = C
            cap_results.append(out)

            pd.DataFrame(cap_results).to_csv(
                cap_path,
                index=False,
            )

cap_df = pd.DataFrame(cap_results)

cap_summary = (
    cap_df
    .groupby(
        ["C", "width", "params"],
        as_index=False,
    )
    .agg(
        nmse_mean=("test_nmse", "mean"),
        nmse_std=("test_nmse", "std"),
    )
)

cap_gap = cap_summary.pivot_table(
    index=["width", "params"],
    columns="C",
    values="nmse_mean",
)

cap_gap["delta_500_minus_50"] = (
    cap_gap[500] - cap_gap[50]
)

print(cap_gap)

cap_gap.to_csv(
    OUT_DIR / "cifar_capacity_gap.csv"
)

Loaded 18 cached capacity rows.
SKIP CAP 50 8 0
SKIP CAP 50 8 1
SKIP CAP 50 8 2
SKIP CAP 50 32 0
SKIP CAP 50 32 1
SKIP CAP 50 32 2
SKIP CAP 50 128 0
SKIP CAP 50 128 1
SKIP CAP 50 128 2
SKIP CAP 500 8 0
SKIP CAP 500 8 1
SKIP CAP 500 8 2
SKIP CAP 500 32 0
SKIP CAP 500 32 1
SKIP CAP 500 32 2
SKIP CAP 500 128 0
SKIP CAP 500 128 1
SKIP CAP 500 128 2
C                   50       500  delta_500_minus_50
width params                                        
8     1611    0.321530  0.551832            0.230302
32    20259   0.318622  0.550978            0.232356
128   302211  0.318836  0.551711            0.232876


# 4D. RAC threshold sensitivity

In [8]:
rho_grid = np.round(
    np.arange(0.10, 0.91, 0.05),
    2,
)

rac_rows = []

for rho in rho_grid:
    for C in [50, 500]:
        sub = (
            cap_df[cap_df["C"] == C]
            .groupby(["width", "params"], as_index=False)
            .agg(
                mean_nmse=("test_nmse", "mean"),
                std_nmse=("test_nmse", "std"),
            )
            .sort_values("params")
        )

        feasible = sub[
            sub["mean_nmse"] <= rho
        ]

        if len(feasible) == 0:
            min_params = np.nan
            achieved = np.nan
        else:
            row = feasible.iloc[0]
            min_params = int(row["params"])
            achieved = float(row["mean_nmse"])

        rac_rows.append({
            "rho": rho,
            "C": C,
            "min_params": min_params,
            "achieved_mean_nmse": achieved,
        })

rac_sensitivity = pd.DataFrame(rac_rows)

rac_pivot = rac_sensitivity.pivot(
    index="rho",
    columns="C",
    values="min_params",
)

print(rac_pivot)

rac_sensitivity.to_csv(
    OUT_DIR / "rac_threshold_sensitivity.csv",
    index=False,
)

C        50      500
rho                 
0.10     NaN     NaN
0.15     NaN     NaN
0.20     NaN     NaN
0.25     NaN     NaN
0.30     NaN     NaN
0.35  1611.0     NaN
0.40  1611.0     NaN
0.45  1611.0     NaN
0.50  1611.0     NaN
0.55  1611.0     NaN
0.60  1611.0  1611.0
0.65  1611.0  1611.0
0.70  1611.0  1611.0
0.75  1611.0  1611.0
0.80  1611.0  1611.0
0.85  1611.0  1611.0
0.90  1611.0  1611.0


## Numerical summary only

This version intentionally avoids all plotting. The CSVs below are the authoritative outputs for closing the empirical analysis; publication figures can be produced later from these CSVs without rerunning any training.

In [9]:
print("\n=== DATA SIZE GAP ===")
print(data_pivot.to_string())

print("\n=== ARCHITECTURE GAP ===")
print(arch_gap.to_string())

print("\n=== CAPACITY GAP ===")
print(cap_gap.to_string())

print("\n=== RAC THRESHOLD SENSITIVITY ===")
print(rac_pivot.to_string())



=== DATA SIZE GAP ===
C              50       500  delta_500_minus_50
n_train                                        
96       0.329442  0.559254            0.229812
192      0.328217  0.556512            0.228296
384      0.321530  0.551832            0.230302

=== ARCHITECTURE GAP ===
C                   50       500  delta_500_minus_50
architecture                                        
dilated       0.298916  0.533599            0.234683
local         0.320659  0.552337            0.231678
resnet        0.309351  0.544975            0.235625

=== CAPACITY GAP ===
C                   50       500  delta_500_minus_50
width params                                        
8     1611    0.321530  0.551832            0.230302
32    20259   0.318622  0.550978            0.232356
128   302211  0.318836  0.551711            0.232876

=== RAC THRESHOLD SENSITIVITY ===
C        50      500
rho                 
0.10     NaN     NaN
0.15     NaN     NaN
0.20     NaN     NaN
0.25     NaN     Na

# 4F. Automatic scientific verdict

In [10]:
verdict = {
    "data_size_gap_positive_all_sizes":
        bool(
            (
                data_pivot["delta_500_minus_50"] > 0
            ).all()
        ),

    "architecture_gap_positive_all_architectures":
        bool(
            (
                arch_gap["delta_500_minus_50"] > 0
            ).all()
        ),

    "capacity_gap_positive_all_widths":
        bool(
            (
                cap_gap["delta_500_minus_50"] > 0
            ).all()
        ),
}

print("=" * 70)
print("NOTEBOOK 4 AUTOMATIC CHECKS")
print("=" * 70)

for k, v in verdict.items():
    print(f"{k}: {v}")

pd.DataFrame([verdict]).to_csv(
    OUT_DIR / "automatic_verdict_cifar.csv",
    index=False,
)

manifest = {
    "experiment": "CIFAR reviewer-killer robustness",
    "seed_base": SEED_BASE,
    "train_sizes": TRAIN_SIZES,
    "data_seeds": DATA_SEEDS,
    "data_width": DATA_WIDTH,
    "architectures": ARCHITECTURES,
    "architecture_seeds": ARCH_SEEDS,
    "architecture_width": ARCH_WIDTH,
    "capacity_widths": CAP_WIDTHS,
    "capacity_seeds": CAP_SEEDS,
    "optimizer": "Adam",
    "learning_rate": LR,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "reference": "DDIM-1000",
    "baselines": [50, 500],
    "device": device,
    "pytorch": torch.__version__,
}

with open(
    OUT_DIR / "cifar_robustness_manifest.json",
    "w",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
    )

print("\nSaved everything to:")
print(OUT_DIR)

NOTEBOOK 4 AUTOMATIC CHECKS
data_size_gap_positive_all_sizes: True
architecture_gap_positive_all_architectures: True
capacity_gap_positive_all_widths: True

Saved everything to:
D:\PhD EMP\Paper Writting\STEP_COMPLEXITY\step5_robustness


# What to send back

From `step5_robustness/`, send:

- `cifar_data_size_robustness.csv`
- `cifar_data_size_gap.csv`
- `cifar_architecture_robustness.csv`
- `cifar_architecture_gap.csv`
- `cifar_capacity_robustness.csv`
- `cifar_capacity_gap.csv`
- `rac_threshold_sensitivity.csv`
- `automatic_verdict_cifar.csv`
- `cifar_robustness_manifest.json`

If any cached CSV already exists, the notebook automatically skips those completed jobs.